# 📓 Course Module 2: Non-Contrastive Learning (BYOL)
Welcome to the second module! Here, we explore **BYOL (Bootstrap Your Own Latent)** by Grill et al. (2020).

### Why Non-Contrastive?
Contrastive learning (like SimCLR) relies heavily on **negative pairs** to prevent representation collapse (i.e., the network outputting a constant vector for all inputs). However, negative pairs require massive batch sizes or memory banks.

**BYOL proves that we don't need negative pairs at all!**

### How BYOL avoids collapse without negatives:
1. **Asymmetric Architecture:** The **Online network** has an extra **Predictor** head ($q_\theta$), while the **Target network** does not.
2. **Target Network Updating:** The target network weights $\xi$ are not updated by gradient descent; they are updated as an **Exponential Moving Average (EMA)** of the online network weights $\theta$.
3. **Stop-Gradient:** Gradients flow *only* through the online network.

![BYOL Architecture](https://miro.medium.com/v2/resize:fit:1400/1*y6v23pW_W1W7Bv0JpG8KFA.png)
*(Image representation of BYOL: Image $x$ is augmented into views $v$ and $v'$. View $v$ passes through Online Encoder + Projector + Predictor. View $v'$ passes through Target Encoder + Projector with stop-gradient. The loss minimizes distance between predicted online representation and target projection.)*

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. Data Augmentations
BYOL uses standard visual augmentations, similar to SimCLR, producing two views $v$ and $v'$ for each image.

In [ ]:
class BYOLTransformations:
    def __init__(self, base_transforms):
        self.base_transforms = base_transforms

    def __call__(self, x):
        v1 = self.base_transforms(x)
        v2 = self.base_transforms(x)
        return v1, v2

byol_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

### 2. Network Modules: MLP Head
BYOL introduces specific MLP modules for the Projector ($g$) and the Predictor ($q$).

In [ ]:
class MLPHead(nn.Module):
    """ 2-layer MLP head used for both Projection and Prediction """
    def __init__(self, in_dim, hidden_dim=256, out_dim=128):
        super(MLPHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)

### 3. The BYOL Architecture & Exponential Moving Average (EMA)
Here we define the core BYOL class containing both **Online** and **Target** networks, alongside the target EMA update logic.

In [ ]:
def create_cifar_resnet():
    resnet = models.resnet18(weights=None)
    resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    resnet.maxpool = nn.Identity()
    resnet.fc = nn.Identity()
    return resnet

class BYOL(nn.Module):
    def __init__(self, projection_dim=128, hidden_dim=256, target_decay=0.99):
        super(BYOL, self).__init__()
        self.target_decay = target_decay

        # --- ONLINE NETWORK ---
        self.online_encoder = create_cifar_resnet()
        self.online_projector = MLPHead(512, hidden_dim, projection_dim)
        # The Predictor head is ONLY present in the Online network!
        self.online_predictor = MLPHead(projection_dim, hidden_dim, projection_dim)

        # --- TARGET NETWORK ---
        # Target network is initially a clone of the Online network (Encoder + Projector)
        self.target_encoder = create_cifar_resnet()
        self.target_projector = MLPHead(512, hidden_dim, projection_dim)

        # Initialize Target weights with Online weights
        self.update_target_network(tau=0.0) # tau=0 means copy online directly

        # Freeze gradients for Target network
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        for param in self.target_projector.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def update_target_network(self, tau=None):
        """
        Exponential Moving Average (EMA) update:
        target_weight = tau * target_weight + (1 - tau) * online_weight
        """
        if tau is None:
            tau = self.target_decay

        for online_p, target_p in zip(self.online_encoder.parameters(), self.target_encoder.parameters()):
            target_p.data = tau * target_p.data + (1 - tau) * online_p.data

        for online_p, target_p in zip(self.online_projector.parameters(), self.target_projector.parameters()):
            target_p.data = tau * target_p.data + (1 - tau) * online_p.data

    def forward_online(self, x):
        h = self.online_encoder(x)
        z = self.online_projector(h)
        p = self.online_predictor(z)
        return h, p

    @torch.no_grad()
    def forward_target(self, x):
        h = self.target_encoder(x)
        z = self.target_projector(h)
        return z.detach()

byol_model = BYOL().to(device)
print(byol_model)

### 4. Mean Squared Error Loss on Normalized Predictions
BYOL minimizes the mean squared error between the normalized online prediction $p_\theta$ and the target projection $z_\xi$.

$$\mathcal{L} = 2 - 2 \cdot \frac{\langle p_\theta, z_\xi \rangle}{\|p_\theta\|_2 \cdot \|z_\xi\|_2}$

Notice how this loss is calculated on single views without comparing against negative samples!

In [ ]:
def byol_loss_fn(p, z):
    """
    p: Online predictor output [Batch_size, Dim]
    z: Target projector output [Batch_size, Dim] (already detached/stop-gradient)
    """
    # L2-normalize prediction and projection vectors
    p_norm = F.normalize(p, dim=-1, p=2)
    z_norm = F.normalize(z, dim=-1, p=2)

    # Mean squared error on normalized vectors is equivalent to 2 - 2 * cosine_similarity
    loss = 2 - 2 * (p_norm * z_norm).sum(dim=-1)
    return loss.mean()

---
## 🧪 5. Real Dataset Pre-training & Evaluation (CIFAR-10)

Now we put BYOL to work on CIFAR-10:
1. **Unlabeled Pre-training (100% Data):** Pre-train `BYOL` using both views, calculating the symmetrized loss ($\mathcal{L}_{12} + \mathcal{L}_{21}$) and updating target parameters via EMA.
2. **Linear Probing Evaluation (10% Labeled Data):** Freeze `byol_model.online_encoder`, train a linear classifier on 10% labeled CIFAR-10 data, and compute final downstream classification accuracy on the test set.

In [ ]:
# --- Datasets and DataLoaders Setup ---
unlabeled_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=BYOLTransformations(byol_transform))
unlabeled_loader = DataLoader(unlabeled_trainset, batch_size=128, shuffle=True, num_workers=2)

standard_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

labeled_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=standard_transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

np.random.seed(42)
indices = np.random.choice(len(labeled_trainset), size=5000, replace=False)
labeled_subset = Subset(labeled_trainset, indices)

labeled_loader = DataLoader(labeled_subset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
# --- Step 1: Pre-training BYOL on 100% Unlabeled Data (Symmetrized Loss) ---
optimizer_byol = torch.optim.Adam(byol_model.parameters(), lr=1e-3)

print("Pre-training BYOL model on 100% UNLABELED CIFAR-10 (5 epochs demo)...")
byol_model.train()
for epoch in range(5):
    running_loss = 0.0
    for (v1, v2), _ in unlabeled_loader:
        v1, v2 = v1.to(device), v2.to(device)
        
        # Pass 1: v1 through Online, v2 through Target
        _, p1 = byol_model.forward_online(v1)
        z2_target = byol_model.forward_target(v2)
        loss_12 = byol_loss_fn(p1, z2_target)
        
        # Pass 2: v2 through Online, v1 through Target
        _, p2 = byol_model.forward_online(v2)
        z1_target = byol_model.forward_target(v1)
        loss_21 = byol_loss_fn(p2, z1_target)
        
        # Symmetrized total loss
        total_loss = loss_12 + loss_21
        
        optimizer_byol.zero_grad()
        total_loss.backward()
        optimizer_byol.step()
        
        # Target EMA update
        byol_model.update_target_network(tau=0.99)
        running_loss += total_loss.item()
        
    print(f"  Epoch {epoch+1}/5 - BYOL Loss: {running_loss / len(unlabeled_loader):.4f}")

In [ ]:
# --- Step 2: Linear Probing Evaluation on 10% Labeled Data ---
encoder = byol_model.online_encoder
for param in encoder.parameters():
    param.requires_grad = False  # FREEZE ONLINE ENCODER

classifier = nn.Linear(512, 10).to(device)
optimizer_linear = torch.optim.Adam(classifier.parameters(), lr=1e-2)
ce_loss_fn = nn.CrossEntropyLoss()

print("Training Linear Classifier on 10% Labeled CIFAR-10 (Frozen Online Encoder)...")
encoder.eval()
classifier.train()
for epoch in range(10):
    for inputs, targets in labeled_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        with torch.no_grad():
            features = encoder(inputs)
            
        outputs = classifier(features)
        loss = ce_loss_fn(outputs, targets)
        
        optimizer_linear.zero_grad()
        loss.backward()
        optimizer_linear.step()

# Evaluate downstream accuracy
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100.0 * correct / total

byol_eval_model = nn.Sequential(encoder, classifier)
byol_acc = evaluate(byol_eval_model, test_loader)
print(f"➡️ BYOL + Linear Probe Test Accuracy (10% labels): {byol_acc:.2f}%")